# Flexibility impacts in the 2030/2040 future scenarios

This notebook analyses the role of demand-side flexibility in the 2030/2040 **flex_off** and **flex_on** networks. We focus on how flexible operation of electric vehicles, thermal storage in the heating sector, and battery storage in the power sector changes demand profiles, dispatch patterns, prices, and curtailment across selected European countries. The analysis uses solved PyPSA networks and aggregates results at country level for a small set of representative countries (DE, NL, IT, PL, CZ, GR).
The notebook covers the following:

- BEVs in the transport sector.
- Thermal in the heating sector.
- Battery storage in the power sector.
- Compare locational marginal prices between flex_off and flex_on.
- Generation dispatch by country and technology.
- Analyse storage state of charge and dispatch by technology.
- Utilisation rates of fossil fuel plants.
- curtailment of wind, solar and hydro by country and technology.


## Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

## Paths and nomenclature

In [ ]:
# Define base folders and names
nf_dir = r"C:\Users\genel\Desktop\WIP\OET\00. Network files"
checks_dir = r"C:\Users\genel\Desktop\WIP\OET\01. Checks"
analysis_pre_dir = r"C:\Users\genel\Desktop\WIP\OET\02. Analysis\01. Pre-analysis"
analysis_post_dir = r"C:\Users\genel\Desktop\WIP\OET\02. Analysis\02. Post-analysis"

# Define scenario, climate year names, and possible target years
hf_scen_name = "highflex"
lf_scen_name = "lowflex"
ct_cy_name = "central"
df_cy_name = "dunkelflaute"
vr_cy_name = "vres"
tys = [2030, 2040]

# Function to select path of a specific network file
def select_nf(scen_name, cy_name, ty, n_cache=None, dir_cache=None):

    subfolder_name = f"{scen_name}_{ty}_{cy_name}"
    subfolder_path = Path(nf_dir) / subfolder_name / "networks"
    subfiles = [p for p in subfolder_path.iterdir() if p.is_file()]
    nf_path = subfiles[0]

    if nf_path == dir_cache:
        print("Using cached network file...")
        n = n_cache
    else:
        n = pypsa.Network(nf_path)
        n_cache = n
        dir_cache = nf_path

    return n, n_cache, dir_cache

# Define dictionary of countries and their corresponding nodes (HV electricity)
country_dic = {"AL": ["AL2 0ACAC"],
               "AT": ["AT2 0ACAC"],
               "BA": ["BA2 0ACAC"],
               "BE": ["BE2 0ACAC"],
               "BG": ["BG2 0ACAC"],
               "CH": ["CH2 0ACAC"],
               "CZ": ["CZ2 0ACAC"],
               "DE": ["DE2 0ACAC"],
               "DK": ["DK2 0ACAC"],
               "EE": ["EE2 0ACAC"],
               "ES": ["ES2 0ACAC"],
               "FI": ["FI0 0ACAC"],
               "FR": ["FR2 0ACAC"],
               "GR": ["GR2 0ACAC"],
               "HR": ["HR2 0ACAC"],
               "HU": ["HU2 0ACAC"],
               "IE": ["IE3 0ACAC"],
               "IT": ["IT2 0ACAC"],
               "LT": ["LT2 0ACAC"],
               "LU": ["LU2 0ACAC"],
               "LV": ["LV2 0ACAC"],
               "ME": ["ME2 0ACAC"],
               "MK": ["MK2 0ACAC"],
               "NL": ["NL2 0ACAC"],
               "NO": ["NO0 0ACAC"],
               "PL": ["PL2 0ACAC"],
               "PT": ["PT2 0ACAC"],
               "RO": ["RO2 0ACAC"],
               "RS": ["RS2 0ACAC"],
               "SE": ["SE0 0ACAC"],
               "SI": ["SI2 0ACAC"],
               "SK": ["SK2 0ACAC"],
               "UK": ["GB1 0ACAC"]
               }
all_countries = list(country_dic.keys())
focus_countries = ["DE", "NL", "IT", "PL", "CZ", "GR"]

# List of buses/carriers of interest for overview
focus_buses = ["HV_electricity", "LV_electricity", "Transport", "Heat", "H2"]

## CHECKS: Raw export to Excel and CSV
Export network file(s) to Excel and CSV formats, sampling timeseries at one week per season, for backup checks material.

Check for ENS magnitude and distribution (if activated in the network file)

In [ ]:
def make_it_excel_readable(scen_name, cy_name, ty, include_legacy: bool = False, n_cache=None, dir_cache=None):

    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)
    n_name = f"{scen_name}_{ty}_{cy_name}"

    export_dir = Path(checks_dir) / n_name
    export_dir.mkdir(exist_ok=True)

    if include_legacy:
        # Export to Excel with the predefined function
        n.export_to_excel(f"{export_dir}/nf_legacy_{n_name}.xlsx")

    sample_weeks = {"winter": 2,
                    "spring": 15,
                    "summer": 28,
                    "autumn": 41}

    # Buses
    buses_df = pd.DataFrame(n.buses)
    buses_df.to_csv(f"{export_dir}/buses_{n_name}.csv")
    if len(n.buses_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.buses_t.keys():
                buses_t_df = pd.DataFrame(n.buses_t[param])
                int_len_h = (buses_t_df.index[1] - buses_t_df.index[0]).total_seconds() / 3600
                sample_chunk_len = int(168 / int_len_h + 1)
                buses_t_df = buses_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                buses_t_df.to_csv(f"{export_dir}/t_buses_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for buses in {n_name} scenario.")

    # Carriers
    carriers_df = pd.DataFrame(n.carriers)
    carriers_df.to_csv(f"{export_dir}/carriers_{n_name}.csv")
    if len(n.carriers_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.carriers_t.keys():
                carriers_t_df = pd.DataFrame(n.carriers_t[param])
                int_len_h = int((carriers_t_df.index[1] - carriers_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                carriers_t_df = carriers_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                carriers_t_df.to_csv(f"{export_dir}/t_carriers_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for carriers in {n_name} scenario.")

    # Generators
    generators_df = pd.DataFrame(n.generators)
    generators_df.to_csv(f"{export_dir}/generators_{n_name}.csv")
    if len(n.generators_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.generators_t.keys():
                generators_t_df = pd.DataFrame(n.generators_t[param])
                int_len_h = int((generators_t_df.index[1] - generators_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                generators_t_df = generators_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                generators_t_df.to_csv(f"{export_dir}/t_generators_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for generators in {n_name} scenario.")

    # Global constraints
    global_constraints_df = pd.DataFrame(n.global_constraints)
    global_constraints_df.to_csv(f"{export_dir}/global_constraints_{n_name}.csv")
    if len(n.global_constraints_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.global_constraints_t.keys():
                global_constraints_t_df = pd.DataFrame(n.global_constraints_t[param])
                int_len_h = (global_constraints_t_df.index[1] - global_constraints_t_df.index[0]).total_seconds() / 3600
                sample_chunk_len = int(168 / int_len_h + 1)
                global_constraints_t_df = global_constraints_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                global_constraints_t_df.to_csv(f"{export_dir}/t_global_constraints_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for global constraints in {n_name} scenario.")

    # Lines
    lines_df = pd.DataFrame(n.lines)
    lines_df.to_csv(f"{export_dir}/lines_{n_name}.csv")
    if len(n.lines_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.lines_t.keys():
                lines_t_df = pd.DataFrame(n.lines_t[param])
                int_len_h = int((lines_t_df.index[1] - lines_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                lines_t_df = lines_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                lines_t_df.to_csv(f"{export_dir}/t_lines_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for lines in {n_name} scenario.")

    # Links
    links_df = pd.DataFrame(n.links)
    links_df.to_csv(f"{export_dir}/links_{n_name}.csv")
    if len(n.links_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.links_t.keys():
                links_t_df = pd.DataFrame(n.links_t[param])
                int_len_h = int((links_t_df.index[1] - links_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                links_t_df = links_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                links_t_df.to_csv(f"{export_dir}/t_links_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for links in {n_name} scenario.")

    # Loads
    loads_df = pd.DataFrame(n.loads)
    loads_df.to_csv(f"{export_dir}/loads_{n_name}.csv")
    if len(n.loads_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.loads_t.keys():
                loads_t_df = pd.DataFrame(n.loads_t[param])
                int_len_h = int((loads_t_df.index[1] - loads_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                loads_t_df = loads_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                loads_t_df.to_csv(f"{export_dir}/t_loads_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for loads in {n_name} scenario.")

    # Storage units
    storage_units_df = pd.DataFrame(n.storage_units)
    storage_units_df.to_csv(f"{export_dir}/storage_units_{n_name}.csv")
    if len(n.storage_units_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.storage_units_t.keys():
                storage_units_t_df = pd.DataFrame(n.storage_units_t[param])
                int_len_h = int((storage_units_t_df.index[1] - storage_units_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                storage_units_t_df = storage_units_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                storage_units_t_df.to_csv(f"{export_dir}/t_storage_units_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for storage units in {n_name} scenario.")

    # Stores
    stores_df = pd.DataFrame(n.stores)
    stores_df.to_csv(f"{export_dir}/stores_{n_name}.csv")
    if len(n.stores_t.keys())>0:
        for season, w in sample_weeks.items():
            for param in n.stores_t.keys():
                stores_t_df = pd.DataFrame(n.stores_t[param])
                int_len_h = int((stores_t_df.index[1] - stores_t_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                stores_t_df = stores_t_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                stores_t_df.to_csv(f"{export_dir}/t_stores_{param}_{n_name}_w_{season}.csv")
    else:
        print(f"No time series data for stores in {n_name} scenario.")

    return n_cache, dir_cache


def check_ens(scen_name, cy_name, ty, n_cache=None, dir_cache=None):

    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)
    n_name = f"{scen_name}_{ty}_{cy_name}"

    export_dir = Path(checks_dir) / n_name
    export_dir.mkdir(exist_ok=True)

    generators_t_df = pd.DataFrame(n.generators_t["p"])
    ens_mask = generators_t_df.columns.str.contains("load")
    if ens_mask.any():
        ens_t_df = generators_t_df.loc[:, ens_mask]
        int_len_h = int((ens_t_df.index[1] - ens_t_df.index[0]).total_seconds() / 3600)
        sample_chunk_len = int(168 / int_len_h + 1)
        ens_t_df = ens_t_df.round(decimals=1)
        ens_t_df.to_csv(f"{export_dir}/t_ens_{n_name}.csv")
        ens_df = ens_t_df.sum(axis=0) * sample_chunk_len / 1e6
        ens_df = ens_df[ens_df > 1e-6]
        for ens_gen, ens in ens_df.items():
            print(f"{ens_gen}: {ens:.2f} TWh of ENS")
    else:
        print(f"ENS is not enabled in the network file for {n_name} scenario.")
    
    return n_cache, dir_cache

In [ ]:
scen_names = ["lowflex", "highflex"]
tys = [2040]
cy_names = ["central"]
n, n_cache, dir_cache = select_nf(scen_names[1], cy_names[0], tys[0])


df = pd.DataFrame(n.generators_t["p"])

print(df.loc[:, "DE2 0ACAC 0 onwind"].sum() * 3 / (160000 * 8760))

# for cy_name in cy_names:
#     for ty in tys:
#         for scen_name in scen_names:
#             n_cache, dir_cache = make_it_excel_readable(scen_name, cy_name, ty, include_legacy=False)
#             check_ens(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)

## ANALYSIS: Computation and export of results
Aggregate granular data from solved network file(s) and export useful insights
### Conventions
For a bus: 
- Energy entering has positive sign (generation, imports)
- Energy exiting has negative sign (demand, exports)

### Pre-analysis
Define functions to compute energy (especially power) balances

In [ ]:
def get_load_ts_for_bus(n, bus_name):

    # Query name of loads connected to bus, then get their time series
    load_ids = n.loads[n.loads["bus"] == bus_name].index
    if len(load_ids)>0:
        load_ids = list(set(n.loads_t.p.columns).intersection(set(load_ids)))
        load_ts = n.loads_t.p[load_ids] * -1
    else:
        print(f"No loads found for bus {bus_name}")
        load_ts = pd.DataFrame()

    return load_ts

def get_generation_ts_for_bus(n, bus_name):

    # Query name of generators connected to bus, then get their time series
    gen_ids = n.generators[n.generators["bus"] == bus_name].index
    if len(gen_ids)>0:
        gen_ids = list(set(n.generators_t.p.columns).intersection(set(gen_ids)))
        gen_ts = n.generators_t.p[gen_ids]
    else:
        print(f"No generators found for bus {bus_name}")
        gen_ts = pd.DataFrame()

    return gen_ts

def get_storage_units_ts_for_bus(n, bus_name):

    # Query name of storage units connected to bus, then get their time series
    su_ids = n.storage_units[n.storage_units["bus"] == bus_name].index
    if len(su_ids)>0:
        su_ids = list(set(n.storage_units_t.p.columns).intersection(set(su_ids)))
        su_ts = n.storage_units_t.p[su_ids]
    else:
        print(f"No storage units found for bus {bus_name}")
        su_ts = pd.DataFrame()

    return su_ts

def get_stores_ts_for_bus(n, bus_name):

    # Query name of stores connected to bus, then get their time series
    store_ids = n.stores[n.stores["bus"] == bus_name].index
    if len(store_ids)>0:
        store_ids = list(set(n.stores_t.p.columns).intersection(set(store_ids)))
        store_ts = n.stores_t.p[store_ids]
    else:
        print(f"No stores found for bus {bus_name}")
        store_ts = pd.DataFrame()

    return store_ts

def get_lines_ts_for_bus(n, bus_name):

    # Query name of lines connected to bus, then get their time series
    line_ids_out = n.lines[n.lines["bus0"] == bus_name].index
    line_ids_in = n.lines[n.lines["bus1"] == bus_name].index

    if len(line_ids_out)==0 and len(line_ids_in)==0:
        print(f"No lines found for bus {bus_name}")
        line_ts = pd.DataFrame()
    else:
        rename_dict = {}
        line_ids_out = list(set(n.lines_t.p0.columns).intersection(set(line_ids_out)))
        for line_id in line_ids_out:
            rename_dict[line_id] = f"interconnector {n.lines.loc[line_id, "bus0"]} - {n.lines.loc[line_id, "bus1"]}"
        line_ts_out = n.lines_t.p0[line_ids_out] * -1 # By convention, p0 is positive if out of bus0
        line_ids_in = list(set(n.lines_t.p1.columns).intersection(set(line_ids_in)))
        for line_id in line_ids_in:
            rename_dict[line_id] = f"interconnector {n.lines.loc[line_id, "bus1"]} - {n.lines.loc[line_id, "bus0"]}"
        line_ts_in = n.lines_t.p1[line_ids_in] * -1 # By convention, p1 is positive if out of bus1
        concat_line_tss = []
        for line_ts_df in [line_ts_out, line_ts_in]:
            if line_ts_df.empty:
                continue
            else:
                concat_line_tss.append(line_ts_df)

        line_ts = pd.concat(concat_line_tss, axis=1)
        line_ts = line_ts.rename(columns=rename_dict)

    return line_ts

def get_links_ts_for_bus(n, bus_name):

    # Query name of links connected to bus, then get their time series
    link_ids_0 = n.links[n.links["bus0"] == bus_name].index
    link_ids_1 = n.links[n.links["bus1"] == bus_name].index
    link_ids_2 = n.links[n.links["bus2"] == bus_name].index
    link_ids_3 = n.links[n.links["bus3"] == bus_name].index
    link_ids_4 = n.links[n.links["bus4"] == bus_name].index
    if all(len(ids) == 0 for ids in [link_ids_0, link_ids_1, link_ids_2, link_ids_3, link_ids_4]):
        print(f"No links found for bus {bus_name}")
        link_ts = pd.DataFrame()
    else:
        link_ids_0 = list(set(n.links_t.p0.columns).intersection(set(link_ids_0)))
        link_ts_0 = n.links_t.p0[link_ids_0] * -1 # By convention, p0 is positive if out of bus0
        link_ids_1 = list(set(n.links_t.p1.columns).intersection(set(link_ids_1)))
        link_ts_1 = n.links_t.p1[link_ids_1] * -1 # By convention, p1 is positive if out of bus1
        link_ids_2 = list(set(n.links_t.p2.columns).intersection(set(link_ids_2)))
        link_ts_2 = n.links_t.p2[link_ids_2] * -1 # By convention, p2 is positive if out of bus2
        link_ids_3 = list(set(n.links_t.p3.columns).intersection(set(link_ids_3)))
        link_ts_3 = n.links_t.p3[link_ids_3] * -1 # By convention, p3 is positive if out of bus3
        link_ids_4 = list(set(n.links_t.p4.columns).intersection(set(link_ids_4)))
        link_ts_4 = n.links_t.p4[link_ids_4] * -1 # By convention, p4 is positive if out of bus4
        concat_link_tss = []
        for link_ts_df in [link_ts_0, link_ts_1, link_ts_2, link_ts_3, link_ts_4]:
            if link_ts_df.empty:
                continue
            else:
                concat_link_tss.append(link_ts_df)

        link_ts = pd.concat(concat_link_tss, axis=1)

    return link_ts

# Put all the above together
def ts_for_bus_to_csv(n, bus_name, export_dir, return_ts=False, explicit=False):

    # Prepare export subdirectory
    export_subdir = export_dir / f"Dispatch balance TS - {bus_name}"
    export_subdir.mkdir(exist_ok=True)

    # Extract time series for the indicated bus
    load_ts = get_load_ts_for_bus(n, bus_name)
    gen_ts = get_generation_ts_for_bus(n, bus_name)
    su_ts = get_storage_units_ts_for_bus(n, bus_name)
    store_ts = get_stores_ts_for_bus(n, bus_name)
    line_ts = get_lines_ts_for_bus(n, bus_name)
    link_ts = get_links_ts_for_bus(n, bus_name)
    ts_dic = {"loads": load_ts,
              "generators": gen_ts,
              "storage_units": su_ts,
              "stores": store_ts,
              "lines": line_ts,
              "links": link_ts
              }

    # Export individual time series to CSV
    for ts_name, ts_df in ts_dic.items():
        if not ts_df.empty:
            ts_df.to_csv(f"{export_subdir}/ts_{bus_name}_{ts_name}.csv")
        else:
            continue

    # Return combined time series only if requested
    if return_ts:
        ts_dfs = []
        for ts_name, ts_df in ts_dic.items():
            if explicit:
                if isinstance(ts_df, pd.DataFrame) and not ts_df.empty:
                    ts_df = ts_df.add_suffix(f" - {ts_name}")
                elif isinstance(ts_df, pd.Series) and not ts_df.empty:
                    ts_df = ts_df.rename(f"{ts_df.name} - {ts_name}")
            ts_dfs.append(ts_df)
        # Combine dfs for granular balance
        ts_df = pd.concat(ts_dfs, axis=1)
        return ts_df
    else:
        return None

Define functions to extract marginal prices for each bus (especially power)

In [ ]:
def get_mp_ts_for_bus(n, bus_name):

    # Query marginal price time-series for selected bus
    mp_ts = n.buses_t.marginal_price[[bus_name]]
    mp_ts.rename(columns={bus_name: "mp"}, inplace=True)

    return mp_ts

# Collect and export marginal price time-series for each bus of interest
def ts_for_node_mp(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None, look_for_backup=False):

    # Load network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)
    n_name = f"{scen_name}_{ty}_{cy_name}"

    # Prepare export directories
    export_dir = Path(analysis_pre_dir) / n_name
    export_dir.mkdir(exist_ok=True)
    export_subdir = export_dir / "Marginal prices TSs"
    export_subdir.mkdir(exist_ok=True)

    # Create dictionary for storing country-level datasets
    ts_mps_country = dict()
    for node in country_dic[country]:
        # Create dictionary for storing node-level datasets
        ts_mps_node = dict()

        # Create paths for aggregated time-series CSVs, specific to node and bus
        hv_mp_path = f"{export_subdir}/ts_mp_{node}_hv_electricity.csv"
        lv_mp_path = f"{export_subdir}/ts_mp_{node}_lv_electricity.csv"
        heat_mp_path = f"{export_subdir}/ts_mp_{node}_heat.csv"
        h2_mp_path = f"{export_subdir}/ts_mp_{node}_h2.csv"

        # HV electricity bus
        if look_for_backup and Path(hv_mp_path).is_file():
            ts_mps_node["HV_electricity"] = pd.read_csv(hv_mp_path, index_col=0, parse_dates=True)
        else:
            hv_bus_name = node
            ts_hv_raw = get_mp_ts_for_bus(n, hv_bus_name)
            ts_hv_raw.to_csv(hv_mp_path)
            ts_mps_node["HV_electricity"] = ts_hv_raw

        # LV electricity bus
        if look_for_backup and Path(lv_mp_path).is_file():
            ts_mps_node["LV_electricity"] = pd.read_csv(lv_mp_path, index_col=0, parse_dates=True)
        else:
            lv_bus_name = f"{node} low voltage"
            ts_lv_raw = get_mp_ts_for_bus(n, lv_bus_name)
            ts_lv_raw.to_csv(lv_mp_path)
            ts_mps_node["LV_electricity"] = ts_lv_raw

        # Heat bus
        if look_for_backup and Path(heat_mp_path).is_file():
            ts_mps_node["Heat"] = pd.read_csv(heat_mp_path, index_col=0, parse_dates=True)
        else:
            ts_heat_raws = []
            for heat_bus_type in ["urban decentral heat", "urban central heat", "rural heat"]:
                heat_bus_name = f"{node} {heat_bus_type}"
                ts_heat_raw = get_mp_ts_for_bus(n, heat_bus_name)
                ts_heat_raws.append(ts_heat_raw)
            ts_heat_raw = pd.concat(ts_heat_raws, axis=1)
            ts_heat_raw.to_csv(heat_mp_path)
            ts_mps_node["Heat"] = ts_heat_raw

        # H2 bus
        if look_for_backup and Path(h2_mp_path).is_file():
            ts_mps_node["H2"] = pd.read_csv(h2_mp_path, index_col=0, parse_dates=True)
        else:
            h2_bus_name = f"{node} H2"
            ts_h2_raw = get_mp_ts_for_bus(n, h2_bus_name)
            ts_h2_raw.to_csv(h2_mp_path)
            ts_mps_node["H2"] = ts_h2_raw

        ts_mps_country[node] = ts_mps_node

    return ts_mps_country, n_cache, dir_cache

In [ ]:
fuel_bus_dict = {"oil_p": "EU oil primary",
                 "oil": "EU oil",
                 "coal": "EU coal",
                 "lignite": "EU lignite",
                 "gas": "EU gas",
                 "biogas": "EU biogas",
                 "methanol": "EU methanol",
                 "uranium": "EU uranium"
                 }

def ts_for_fuel_agg(scen_name, cy_name, ty, n_cache=None, dir_cache=None, look_for_backup=False):

    # Load network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)
    n_name = f"{scen_name}_{ty}_{cy_name}"

    # Prepare export directories
    export_dir = Path(analysis_pre_dir) / n_name
    export_dir.mkdir(exist_ok=True)
    export_subdir = export_dir / "Aggregated fuel demand TSs"
    export_subdir.mkdir(exist_ok=True)

    ts_aggs_fuels = dict()
    for fuel in fuel_bus_dict.keys():
        # Create paths for aggregated time-series CSVs, specific to fuel
        fuel_agg_path = f"{export_subdir}/ts_agg_{fuel}.csv"
        if look_for_backup and Path(fuel_agg_path).is_file():
            ts_aggs_fuels[fuel] = pd.read_csv(fuel_agg_path, index_col=0, parse_dates=True)
        else:
            fuel_bus_name = fuel_bus_dict[fuel]
            ts_fuel_raw = ts_for_bus_to_csv(n, fuel_bus_name, export_dir, return_ts=True)
            ts_aggs_fuels[fuel] = ts_fuel_raw

    return ts_aggs_fuels, n_cache, dir_cache

Define conventions and functions to aggregate power balances for power (HV and LV), heat (rural, urban centralized and decentralized) and hydrogen

In [ ]:
# Dictionary for categorizing HV power components
rename_balance_p_hv = {"offwind": "gen_wind_offshore",
                       "onwind": "gen_wind_onshore",
                       "solar": "gen_solar_pv",
                       "ror": "gen_hydro_ror",
                       "hydro": "gen_hydro_reservoir",
                       "PHS": "gen_hydro_phs",
                       "battery": "gen_battery",
                       "nuclear": "gen_nuclear",
                       "H2 Fuel Cell": "gen_h2",
                       "H2 turbine": "gen_h2",
                       "biomass_chp": "gen_biomass_chp",
                       "gas_chp": "gen_methane_chp",
                       "CCGT": "gen_methane",
                       "OCGT": "gen_methane",
                       "coal_chp": "gen_coal_chp",
                       "lignite_chp": "gen_coal_chp",
                       "lignite ": "gen_coal",
                       "coal ": "gen_coal",
                       "interconnector": "grid_interconnectors",
                       "relation": "grid_interconnectors",
                       "H2 pipeline": "grid_h2",
                       "methanolisation": "grid_methanol",
                       "distribution grid": "load_distribution",
                       "Haber-Bosch": "load_p2h",
                       "H2 Electrolysis": "load_p2h",
                       "DAC": "load_dac",
                       # These entries will match exactly the name of the component
                       "node_name load": "gen_ens"
                       }

# Dictionary for categorizing LV power components
rename_balance_p_lv = {"distribution grid": "gen_transmission",
                       "solar rooftop": "gen_solar_btm",
                       "V2G": "gen_ev",
                       "home battery": "gen_battery_btm",
                       "BEV charger": "load_ev",
                       "heat pump": "load_hp",
                       "resistive heater": "load_rh",
                       # These entries will match exactly the name of the component
                       "node_name low voltage load": "gen_ens",
                       "node_name": "load_baseline",
                       "node_name industry electricity": "load_industry"
                       }

# Dictionary for categorizing LV power components
rename_balance_trans = {"BEV charger - links": "gen_bev_charger",
                        "EV battery - stores": "gen_bev",
                        "land transport oil - links": "gen_oil",
                        "V2G - links": "load_bev_charger",
                        "land transport EV - loads": "load_bev",
                        "land transport oil - loads": "load_oil"
                        }

# Dictionary for categorizing heat components
rename_balance_heat = {"solar thermal collector": "gen_solar_thermal",
                       "heat pump": "gen_hp",
                       "resistive heater": "gen_rh",
                       "biomass_chp": "gen_biomass_chp",
                       "biomass boiler": "gen_biomass",
                       "gas_chp": "gen_methane_chp",
                       "gas boiler": "gen_methane",
                       "coal_chp": "gen_coal_chp",
                       "lignite_chp": "gen_coal_chp",
                       "Fischer-Tropsch": "gen_byproduct",
                       "H2 Electrolysis": "gen_byproduct",
                       "H2 Fuel Cell": "gen_byproduct",
                       "Haber-Bosch": "gen_byproduct",
                       "methanolisation": "gen_byproduct",
                       "Sabatier": "gen_byproduct",
                       "DAC": "gen_byproduct",
                       "heat dsm": "gen_dsm",
                       "water tanks": "gen_wtanks",
                       "water pits": "gen_wpits",
                       "heat vent": "load_ventilation",
                       # These entries will match exactly the name of the component
                       "node_name urban central heat load": "gen_ens",
                       "node_name urban decentral heat load": "gen_ens",
                       "node_name rural heat load": "gen_ens",
                       "node_name low-temperature heat for industry": "load_industry",
                       "node_name urban central heat": "load_baseline",
                       "node_name urban decentral heat": "load_baseline",
                       "node_name rural heat": "load_baseline"
                       }

# Dictionary for categorizing H2 components
rename_balance_h2 = {"Haber-Bosch": "gen_p2h",
                     "H2 Electrolysis": "gen_p2h",
                     "ammonia cracker": "gen_cracking",
                     "SMR": "gen_smr",
                     "H2 pipeline": "grid_interconnectors",
                     "Fischer-Tropsch": "load_fossil",
                     "Sabatier": "load_fossil",
                     "methanolisation": "load_fossil",
                     "electrobiofuels": "load_fossil",
                     "H2 Fuel Cell": "load_power",
                     "H2 turbine": "load_power",
                     # These entries will match exactly the name of the component
                     "node_name H2 load": "gen_ens",
                     "node_name H2 for industry": "load_industry"
                     }

def ts_for_node_agg(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None, look_for_backup=False):

    # Load network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)
    n_name = f"{scen_name}_{ty}_{cy_name}"

    # Prepare export directories
    export_dir = Path(analysis_pre_dir) / n_name
    export_dir.mkdir(exist_ok=True)
    export_subdir = export_dir / "Aggregated dispatch balance TSs"
    export_subdir.mkdir(exist_ok=True)

    # Create dictionary for storing country-level datasets
    ts_aggs_country = dict()
    for node in country_dic[country]:
        # Create dictionary for storing node-level datasets
        ts_aggs_node = dict()

        # Create paths for aggregated time-series CSVs, specific to node and bus
        hv_agg_path = f"{export_subdir}/ts_agg_{node}_hv_electricity.csv"
        lv_agg_path = f"{export_subdir}/ts_agg_{node}_lv_electricity.csv"
        trans_agg_path = f"{export_subdir}/ts_agg_{node}_transport.csv"
        heat_agg_path = f"{export_subdir}/ts_agg_{node}_heat.csv"
        h2_agg_path = f"{export_subdir}/ts_agg_{node}_h2.csv"

        # HV electricity bus
        if look_for_backup and Path(hv_agg_path).is_file():
            ts_aggs_node["HV_electricity"] = pd.read_csv(hv_agg_path, index_col=0, parse_dates=True)
        else:
            hv_bus_name = node
            ts_hv_raw = ts_for_bus_to_csv(n, hv_bus_name, export_dir, return_ts=True)
            ts_hv_aggs = []
            for category in rename_balance_p_hv.keys():
                mask = category.replace("node_name", node) if "node_name" in category else ts_hv_raw.columns.str.contains(category)
                if isinstance(mask, str) and mask not in ts_hv_raw.columns:
                    continue
                category_df = ts_hv_raw.loc[:, mask]
                if not category_df.empty:
                    if isinstance(category_df, pd.Series):
                        category_df = category_df.to_frame()
                    category_df.columns = [rename_balance_p_hv[category]] * len(category_df.columns)
                    ts_hv_aggs.append(category_df)
                else:
                    continue
            if len(ts_hv_aggs)>0:
                ts_hv_agg = pd.concat(ts_hv_aggs, axis=1)
                ts_hv_agg = ts_hv_agg.T.groupby(level=0).sum().T
                # Check that the balance is actually a balance, if not raise an error or at least a warning
                if abs(ts_hv_agg.sum(axis=1).max()) > 1:
                    ts_hv_agg.to_csv(hv_agg_path.replace(".csv", "_debug.csv"))
                    raise ValueError(f"{country} HV balance is not balancing, check aggregation dictionary")
                elif abs(ts_hv_agg.sum(axis=1).max()) > 0.01:
                    print(f"{country} HV balance can be off by more than 0.01 MW")
                ts_hv_agg.to_csv(hv_agg_path)
                ts_aggs_node["HV_electricity"] = ts_hv_agg

        # LV electricity bus
        if look_for_backup and Path(lv_agg_path).is_file():
            ts_aggs_node["LV_electricity"] = pd.read_csv(lv_agg_path, index_col=0, parse_dates=True)
        else:
            lv_bus_name = f"{node} low voltage"
            ts_lv_raw = ts_for_bus_to_csv(n, lv_bus_name, export_dir, return_ts=True)
            ts_lv_aggs = []
            for category in rename_balance_p_lv.keys():
                mask = category.replace("node_name", node) if "node_name" in category else ts_lv_raw.columns.str.contains(category)
                if isinstance(mask, str) and mask not in ts_lv_raw.columns:
                    continue
                category_df = ts_lv_raw.loc[:, mask]
                if not category_df.empty:
                    if isinstance(category_df, pd.Series):
                        category_df = category_df.to_frame()
                    category_df.columns = [rename_balance_p_lv[category]] * len(category_df.columns)
                    ts_lv_aggs.append(category_df)
                else:
                    continue
            if len(ts_lv_aggs)>0:
                ts_lv_agg = pd.concat(ts_lv_aggs, axis=1)
                ts_lv_agg = ts_lv_agg.T.groupby(level=0).sum().T
                # Check that the balance is actually a balance, if not raise an error or at least a warning
                if abs(ts_lv_agg.sum(axis=1).max()) > 1:
                    ts_lv_agg.to_csv(lv_agg_path.replace(".csv", "_debug.csv"))
                    raise ValueError(f"{country} LV balance is not balancing, check aggregation dictionary")
                elif abs(ts_lv_agg.sum(axis=1).max()) > 0.01:
                    print(f"{country} LV balance can be off by more than 0.01 MW")
                ts_lv_agg.to_csv(lv_agg_path)
                ts_aggs_node["LV_electricity"] = ts_lv_agg

        # Transport bus(es)
        if look_for_backup and Path(trans_agg_path).is_file():
            ts_aggs_node["Transport"] = pd.read_csv(trans_agg_path, index_col=0, parse_dates=True)
        else:
            ts_trans_raws = []
            for trans_bus_type in ["land transport oil", "EV battery"]:
                trans_bus_name = f"{node} {trans_bus_type}"
                ts_trans_raw = ts_for_bus_to_csv(n, trans_bus_name, export_dir, return_ts=True, explicit=True)
                ts_trans_raws.append(ts_trans_raw)
            if len(ts_trans_raws)>0:
                ts_trans_raw = pd.concat(ts_trans_raws, axis=1)
                ts_trans_aggs = []
                for category in rename_balance_trans.keys():
                    mask = category.replace("node_name", node) if "node_name" in category else ts_trans_raw.columns.str.contains(category)
                    if isinstance(mask, str) and mask not in ts_trans_raw.columns:
                        continue
                    category_df = ts_trans_raw.loc[:, mask]
                    if not category_df.empty:
                        if isinstance(category_df, pd.Series):
                            category_df = category_df.to_frame()
                        category_df.columns = [rename_balance_trans[category]] * len(category_df.columns)
                        ts_trans_aggs.append(category_df)
                    else:
                        continue
                if len(ts_trans_aggs)>0:
                    ts_trans_agg = pd.concat(ts_trans_aggs, axis=1)
                    ts_trans_agg = ts_trans_agg.T.groupby(level=0).sum().T
                    # Check that the balance is actually a balance, if not raise an error or at least a warning
                    if abs(ts_trans_agg.sum(axis=1).max()) > 1:
                        ts_trans_agg.to_csv(trans_agg_path.replace(".csv", "_debug.csv"))
                        raise ValueError(f"{country} transport balance is not balancing, check aggregation dictionary")
                    elif abs(ts_trans_agg.sum(axis=1).max()) > 0.01:
                        print(f"{country} transport balance can be off by more than 0.01 MW")
                    ts_trans_agg.to_csv(trans_agg_path)
                    ts_aggs_node["Transport"] = ts_trans_agg
        
        # Heat bus
        if look_for_backup and Path(heat_agg_path).is_file():
            ts_aggs_node["Heat"] = pd.read_csv(heat_agg_path, index_col=0, parse_dates=True)
        else:
            ts_heat_raws = []
            for heat_bus_type in ["urban decentral heat", "urban central heat", "rural heat"]:
                heat_bus_name = f"{node} {heat_bus_type}"
                ts_heat_raw = ts_for_bus_to_csv(n, heat_bus_name, export_dir, return_ts=True)
                ts_heat_raws.append(ts_heat_raw)
            if len(ts_heat_raws)>0:
                ts_heat_raw = pd.concat(ts_heat_raws, axis=1)
                ts_heat_aggs = []
                for category in rename_balance_heat.keys():
                    mask = category.replace("node_name", node) if "node_name" in category else ts_heat_raw.columns.str.contains(category)
                    if isinstance(mask, str) and mask not in ts_heat_raw.columns:
                        continue
                    category_df = ts_heat_raw.loc[:, mask]
                    if not category_df.empty:
                        if isinstance(category_df, pd.Series):
                            category_df = category_df.to_frame()
                        category_df.columns = [rename_balance_heat[category]] * len(category_df.columns)
                        ts_heat_aggs.append(category_df)
                    else:
                        continue
                if len(ts_heat_aggs)>0:
                    ts_heat_agg = pd.concat(ts_heat_aggs, axis=1)
                    ts_heat_agg = ts_heat_agg.T.groupby(level=0).sum().T
                    # Check that the balance is actually a balance, if not raise an error or at least a warning
                    if abs(ts_heat_agg.sum(axis=1).max()) > 1:
                        ts_heat_agg.to_csv(heat_agg_path.replace(".csv", "_debug.csv"))
                        raise ValueError(f"{country} heat balance is not balancing, check aggregation dictionary")
                    elif abs(ts_heat_agg.sum(axis=1).max()) > 0.01:
                        print(f"{country} heat balance can be off by more than 0.01 MW")
                    ts_heat_agg.to_csv(heat_agg_path)
                    ts_aggs_node["Heat"] = ts_heat_agg

        # H2 bus
        if look_for_backup and Path(h2_agg_path).is_file():
            ts_aggs_node["H2"] = pd.read_csv(h2_agg_path, index_col=0, parse_dates=True)
        else:
            h2_bus_name = f"{node} H2"
            ts_h2_raw = ts_for_bus_to_csv(n, h2_bus_name, export_dir, return_ts=True)
            ts_h2_aggs = []
            for category in rename_balance_h2.keys():
                mask = category.replace("node_name", node) if "node_name" in category else ts_h2_raw.columns.str.contains(category)
                if isinstance(mask, str) and mask not in ts_h2_raw.columns:
                        continue
                category_df = ts_h2_raw.loc[:, mask]
                if not category_df.empty:
                    if isinstance(category_df, pd.Series):
                        category_df = category_df.to_frame()
                    category_df.columns = [rename_balance_h2[category]] * len(category_df.columns)
                    ts_h2_aggs.append(category_df)
                else:
                    continue
            if len(ts_h2_aggs)>0:
                ts_h2_agg = pd.concat(ts_h2_aggs, axis=1)
                ts_h2_agg = ts_h2_agg.T.groupby(level=0).sum().T
                # Check that the balance is actually a balance, if not raise an error or at least a warning
                if abs(ts_h2_agg.sum(axis=1).max()) > 1:
                    ts_h2_agg.to_csv(h2_agg_path.replace(".csv", "_debug.csv"))
                    raise ValueError(f"{country} H2 balance is not balancing, check aggregation dictionary")
                elif abs(ts_h2_agg.sum(axis=1).max()) > 0.01:
                    print(f"{country} H2 balance can be off by more than 0.01 MW")
                ts_h2_agg.to_csv(h2_agg_path)
                ts_aggs_node["H2"] = ts_h2_agg

        ts_aggs_country[node] = ts_aggs_node

    return ts_aggs_country, n_cache, dir_cache

Define functions and conventions to represent power balances for power (HV and LV), heat (rural, urban centralized and decentralized) and hydrogen

In [ ]:
# Dictionary for color coding
balance_colors = {"load_distribution": "#9f1a1a",
                  "load_p2h": "#e43d3d",
                  "load_dac": "#de9191",
                  "load_industry": "#460808",
                  "load_baseline": "#9f1a1a",
                  "load_ev": "#00c8ff",
                  "load_hp": "#d616e0",
                  "load_rh": "#86068d",
                  "load_ventilation": "#de9191",
                  "load_power": "#86068d",
                  "load_fossil": "#4e4e4e",
                  "load_bev_charger": "#00c8ff",
                  "load_bev": "#7bfdff",
                  "load_oil": "#a68500",
                  "gen_wind_offshore": "#8dca00",
                  "gen_wind_onshore": "#b3ff00",
                  "gen_solar_pv": "#ffdd00",
                  "gen_hydro_ror": "#00c8ff",
                  "gen_hydro_reservoir": "#0086c0",
                  "gen_hydro_phs": "#7bfdff",
                  "gen_battery": "#d616e0",
                  "gen_nuclear": "#2aa600",
                  "gen_h2": "#86068d",
                  "gen_biomass_chp": "#004902",
                  "gen_methane_chp": "#a68500",
                  "gen_methane": "#714D13",
                  "gen_coal_chp": "#4e4e4e",
                  "gen_coal": "#141414",
                  "gen_ens": "#9f1a1a",
                  "gen_transmission": "#F17800",
                  "gen_solar_btm": "#ffdd00",
                  "gen_battery_btm": "#8dca00",
                  "gen_ev": "#00c8ff",
                  "gen_solar_thermal": "#ffdd00",
                  "gen_hp": "#d616e0",
                  "gen_rh": "#86068d",
                  "gen_biomass": "#2aa600",
                  "gen_byproduct": "#F17800",
                  "gen_dsm": "#7bfdff",
                  "gen_wtanks": "#00c8ff",
                  "gen_wpits": "#0086c0",
                  "gen_p2h": "#d616e0",
                  "gen_cracking": "#a68500",
                  "gen_smr": "#714D13",
                  "grid_interconnectors": "#F17800",
                  "grid_h2": "#F09539",
                  "grid_methanol": "#F6B06A",
                  "gen_bev_charger": "#00c8ff",
                  "gen_bev": "#7bfdff",
                  "gen_oil": "#714D13"
                  }

# Dictionary for sample weeks
sample_weeks = {"winter": 2,
                "spring": 15,
                "summer": 28,
                "autumn": 41}

# Stacked area charts only accept positive-only or negative-only columns, this function splits the mixed ones
def prepare_df_for_stack(df):

    # Copy color coding dictionary
    colors_dict = balance_colors.copy()

    # Find columns with both positive and negative values
    mixed_mask = (df < 0).any() & (df > 0).any()
    mixed_cols = df.columns[mixed_mask].tolist()
    print(f"Columns with mixed values: {mixed_cols}")
    # Substitute each mixed-value column with two clipped ones (one positive, one negative)
    for col in mixed_cols:
        pos_col = df[col].clip(lower=0)
        neg_col = df[col].clip(upper=0)
        df[f"{col}_pos"] = pos_col
        df[f"{col}_neg"] = neg_col
        df.drop(columns=col, inplace=True)
        if col not in balance_colors.keys():
            raise IndexError(f"'{col}' column is not color-coded")
        color = colors_dict.pop(col)
        colors_dict[f"{col}_pos"] = color
        colors_dict[f"{col}_neg"] = color

    return df, colors_dict

# Function for representing weekly balances and prices
def weekly_dispatch_prices(scen_name, cy_name, ty, country, bus=None, n_cache=None, dir_cache=None):

    if country not in country_dic.keys():
        raise AttributeError

    # Retrieve aggregated time series for focus buses
    ts_aggs_country, n_cache, dir_cache = ts_for_node_agg(scen_name, cy_name, ty, country, look_for_backup=True)
    ts_mps_country = ts_for_node_mp(scen_name, cy_name, ty, country, n_cache, dir_cache, look_for_backup=True)[0]

    # Define buses to be shown
    if bus is None:
        buses = focus_buses
    else:
        buses = [bus]

    # One gridded stacked area chart per node and bus
    for node in country_dic[country]:
        ts_aggs_node = ts_aggs_country[node]
        ts_mps_node = ts_mps_country[node]

        for bus in buses:
            if bus not in ts_aggs_node.keys():
                print(f"{bus} not present in aggregated dispatch timeseries dictionary for node {node}")
                continue
            else:
                dispatch_ts_df, colors_dict = prepare_df_for_stack(ts_aggs_node[bus])
                int_len_h = int((dispatch_ts_df.index[1] - dispatch_ts_df.index[0]).total_seconds() / 3600)
                sample_chunk_len = int(168 / int_len_h + 1)
                chart_title = f"{node} - {bus} weekly dispatch sample"
                fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharey=True)
                fig.suptitle(chart_title, fontsize=16, y=0.95)
                axes = axes.flatten()
                for i, (season, w) in enumerate(sample_weeks.items()):
                    w_dispatch_ts_df = dispatch_ts_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                    og_column_len = len(w_dispatch_ts_df.columns)
                    reorder_columns = [col for col in colors_dict.keys() if col in w_dispatch_ts_df.columns]
                    w_dispatch_ts_df = w_dispatch_ts_df[reorder_columns]
                    # Check that no column is forgotten in reordering
                    if og_column_len > len(w_dispatch_ts_df.columns):
                        raise IndexError(f"Some column from the {node}-{bus} balance dataset is being left out")
                    colors = [colors_dict[col] for col in reorder_columns]
                    ax = axes[i]
                    ax = w_dispatch_ts_df.plot(kind='area', stacked=True, color=colors, ax=ax, legend=False)
                    ax.set_title(season)
                    if i % 2 == 1:  # Only show y-label on right column
                        ax.set_ylabel('Dispatch')
                    else:
                        ax.set_ylabel('')
                handles, labels = axes[0].get_legend_handles_labels()
                fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 0.02), ncol=int(round(len(labels)/4)), fontsize=8)
                plt.tight_layout()
                plt.show()

            if bus not in ts_mps_node.keys():
                print(f"{bus} not present in marginal price timeseries dictionary for node {node}")
                continue
            else:
                mp_ts_df = ts_mps_node[bus]
                sample_chunk_len = int(168 / int_len_h + 1)
                chart_title = f"{node} - {bus} weekly price sample"
                fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharey=True)
                fig.suptitle(chart_title, fontsize=16, y=0.95)
                axes = axes.flatten()
                for i, (season, w) in enumerate(sample_weeks.items()):
                    w_mp_ts_df = mp_ts_df.iloc[(sample_chunk_len*w):(sample_chunk_len*(w+1)), :]
                    ax = axes[i]
                    ax = w_mp_ts_df.plot(kind='line', ax=ax, legend=True)
                    ax.set_title(season)
                    if i % 2 == 1:  # Only show y-label on right column
                        ax.set_ylabel('Marginal price')
                    else:
                        ax.set_ylabel('')
                plt.tight_layout()
                plt.show()

    return None

In [ ]:
scen_name = "lowflex"
cy_name = "central"
ty = 2040
country = "DE"

weekly_dispatch_prices(scen_name, cy_name, ty, country, bus=None)

### Post-analysis
Define functions to compute useful metrics for impact

#### Ember reporting

In [ ]:
# Function to retrieve power capacity from links
def retrieve_elec_obj_capacity(n_df, node, obj_names, capacity_col, divider=1):

    cap = 0
    for obj_name in obj_names:
        try:
            el_cap = n_df.loc[f"{node} {obj_name}", capacity_col] / divider
        except KeyError:
            el_cap = 0
        cap += el_cap
    
    return cap

# Function for putting together key HV metrics for a country
def get_hv_metrics(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None):

    # Retrieve network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)

    if country not in country_dic.keys():
        raise AttributeError

    # Retrieve aggregated time series for focus buses
    ts_aggs_country, n_cache, dir_cache = ts_for_node_agg(scen_name, cy_name, ty, country, n_cache=n_cache, dir_cache=dir_cache, look_for_backup=True)
    ts_mps_country, n_cache, dir_cache = ts_for_node_mp(scen_name, cy_name, ty, country, n_cache=n_cache, dir_cache=dir_cache, look_for_backup=True)

    # Set up list for country concatenation and dictionary of key metrics
    country_km_dfs = []

    # One set of key metrics per node, HV only
    for node in country_dic[country]:

        node_km_dict = {"country": country,
                "node": node,
                "total_dist_demand_twh": 0,
                "avg_dist_demand_gw": 0,
                "std_dev_dist_demand_gw": 0,
                "dist_capacity_gw": 0,
                "total_fossil_gen_twh": 0,
                "total_gas_gen_twh": 0,
                "total_hydro_gen_twh": 0,
                "total_net_import_twh": 0,
                "total_wind_solar_utility_gen_twh": 0,
                "total_wind_solar_utility_curtailment_twh": 0,
                "avg_mp_eur_mwh": 0,
                "std_dev_mp_eur_mwh": 0,
                "utility_bess_capacity_gw": 0,
                "utility_bess_capacity_gwh": 0,
                "btm_bess_capacity_gw": 0,
                "btm_bess_capacity_gwh": 0, 
                "wind_onshore_capacity_gw": 0, 
                "wind_offshore_capacity_gw": 0,
                "utility_solar_capacity_gw": 0,
                "btm_solar_capacity_gw": 0
                }

        node_km_dict["node"] = node
        ts_aggs_node = ts_aggs_country[node]
        ts_mps_node = ts_mps_country[node]

        ts_dispatch = ts_aggs_node["HV_electricity"]
        int_len_h = int((ts_dispatch.index[1] - ts_dispatch.index[0]).total_seconds() / 3600)
        ts_mp = ts_mps_node["HV_electricity"]

        # Load to distribution
        if "load_distribution" in ts_dispatch.columns:
            node_km_dict["total_dist_demand_twh"] = - ts_dispatch["load_distribution"].sum() / 1e6 * int_len_h
            node_km_dict["avg_dist_demand_gw"] = - ts_dispatch["load_distribution"].mean() / 1e3
            node_km_dict["std_dev_dist_demand_gw"] = ts_dispatch["load_distribution"].std() / 1e3
        node_km_dict["dist_capacity_gw"] = n.links.loc[f"{node} electricity distribution grid", "p_nom_opt"] / 1e3

        # Generation and interconnection totals
        fossil_cols = [col for col in ts_dispatch.columns if "coal" in col or "lignite" in col or "methane" in col or "biomass" in col]
        gas_cols = [col for col in ts_dispatch.columns if "methane" in col]
        wind_solar_cols = [col for col in ts_dispatch.columns if "wind" in col or "solar" in col]
        hydro_cols = [col for col in ts_dispatch.columns if "hydro" in col]
        import_cols = [col for col in ts_dispatch.columns if "interconnectors" in col or "relation" in col]
        if fossil_cols:
            node_km_dict["total_fossil_gen_twh"] = ts_dispatch[fossil_cols].sum().sum() / 1e6 * int_len_h
        if gas_cols:
            node_km_dict["total_gas_gen_twh"] = ts_dispatch[gas_cols].sum().sum() / 1e6 * int_len_h
        if hydro_cols:
            node_km_dict["total_hydro_gen_twh"] = ts_dispatch[hydro_cols].sum().sum() / 1e6 * int_len_h
        if import_cols:
            node_km_dict["total_net_import_twh"] = ts_dispatch[import_cols].sum().sum() / 1e6 * int_len_h
        if wind_solar_cols:
            node_km_dict["total_wind_solar_utility_gen_twh"] = ts_dispatch[wind_solar_cols].sum().sum() / 1e6 * int_len_h

        # Curtailment totals
        utility_wind_solar_cols = [f"{node} 0 solar", f"{node} 0 solar-hsat", f"{node} 0 offwind-ac", f"{node} 0 offwind-dc", f"{node} 0 offwind-float", f"{node} 0 onwind"]
        utility_wind_solar_cols = n.generators_t.p.columns.intersection(utility_wind_solar_cols).tolist()
        if not utility_wind_solar_cols:
            raise IndexError(f"No utility-scale wind or solar generator found for node {node}")
        utility_wind_solar_cap = n.generators.loc[n.generators.index.isin(utility_wind_solar_cols), "p_nom_opt"]
        utility_wind_solar_cap_t = n.generators_t.p_max_pu.loc[:, utility_wind_solar_cols] * utility_wind_solar_cap
        node_km_dict["total_wind_solar_utility_curtailment_twh"] = utility_wind_solar_cap_t.sum().sum() / 1e6 * int_len_h - node_km_dict["total_wind_solar_utility_gen_twh"]

        # Marginal price
        node_km_dict["avg_mp_eur_mwh"] = ts_mp["mp"].mean()
        node_km_dict["std_dev_mp_eur_mwh"] = ts_mp["mp"].std()

        # BESS capacities
        node_km_dict["utility_bess_capacity_gw"] = retrieve_elec_obj_capacity(n.storage_units, node, ["battery"], "p_nom_opt", 1e3)
        node_km_dict["utility_bess_capacity_gwh"] = node_km_dict["utility_bess_capacity_gw"] * retrieve_elec_obj_capacity(n.storage_units, node, ["battery"], "max_hours")
        node_km_dict["btm_bess_capacity_gwh"] = retrieve_elec_obj_capacity(n.stores, node, ["home battery"], "e_nom_opt", 1e3)
        node_km_dict["btm_bess_capacity_gw"] = retrieve_elec_obj_capacity(n.links, node, ["home battery charger", "home battery discharger"], "p_nom_opt", 2e3)

        # RES capacities
        node_km_dict["wind_onshore_capacity_gw"] = retrieve_elec_obj_capacity(n.generators, node, ["0 onwind"], "p_nom_opt", 1e3)
        node_km_dict["wind_offshore_capacity_gw"] = retrieve_elec_obj_capacity(n.generators, node, ["0 offwind-ac", "0 offwind-dc", "0 offwind-float"], "p_nom_opt", 1e3)
        node_km_dict["utility_solar_capacity_gw"] = retrieve_elec_obj_capacity(n.generators, node, ["0 solar-hsat", "0 solar"], "p_nom_opt", 1e3) #TODO fix this, it's double counting rooftop solar
        node_km_dict["btm_solar_capacity_gw"] = retrieve_elec_obj_capacity(n.generators, node, ["0 solar rooftop"], "p_nom_opt", 1e3)

        # Put together in DataFrame format for later concatenation at country level
        node_km_df = pd.DataFrame(node_km_dict, index=[0])
        country_km_dfs.append(node_km_df)

    country_km_df = pd.concat(country_km_dfs, axis=0, ignore_index=True)

    return country_km_df, n_cache, dir_cache


# Function for putting together key electricity demand metrics for a country
def get_dm_metrics(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None):

    # Retrieve network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)

    if country not in country_dic.keys():
        raise AttributeError

    # Retrieve aggregated time series for focus buses
    ts_aggs_country, n_cache, dir_cache = ts_for_node_agg(scen_name, cy_name, ty, country, n_cache=n_cache, dir_cache=dir_cache, look_for_backup=True)

    # Set up list for country concatenation and dictionary of demand metrics
    country_dm_dfs = []

    # One set of demand metrics per node
    for node in country_dic[country]:

        node_dm_dict = {"country": country,
                "node": node,
                "total_dist_demand_twh": 0,
                "avg_dist_demand_gw": 0,
                "std_dev_dist_demand_gw": 0,
                "dist_capacity_gw": 0,
                "total_base_demand_twh": 0,
                "total_ind_demand_twh": 0,
                "total_solar_btm_gen_twh": 0,
                "total_ev_demand_twh": 0,
                "total_hp_demand_twh": 0,
                "total_rh_demand_twh": 0
                }

        node_dm_dict["node"] = node
        ts_aggs_node = ts_aggs_country[node]

        # HV electricity bus
        ts_dispatch_hv = ts_aggs_node["HV_electricity"]
        int_len_h = int((ts_dispatch_hv.index[1] - ts_dispatch_hv.index[0]).total_seconds() / 3600)
        # Load to distribution
        if "load_distribution" in ts_dispatch_hv.columns:
            node_dm_dict["total_dist_demand_twh"] = - ts_dispatch_hv["load_distribution"].sum() / 1e6 * int_len_h
            node_dm_dict["avg_dist_demand_gw"] = - ts_dispatch_hv["load_distribution"].mean() / 1e3
            node_dm_dict["std_dev_dist_demand_gw"] = ts_dispatch_hv["load_distribution"].std() / 1e3
        node_dm_dict["dist_capacity_gw"] = n.links.loc[f"{node} electricity distribution grid", "p_nom_opt"] / 1e3

        # LV electricity bus
        ts_dispatch_lv = ts_aggs_node["LV_electricity"]
        # Distribution demand decomposition
        base_cols = [col for col in ts_dispatch_lv.columns if "_baseline" in col]
        ind_cols = [col for col in ts_dispatch_lv.columns if "_industry" in col]
        solar_btm_cols = [col for col in ts_dispatch_lv.columns if "_solar_btm" in col]
        ev_cols = [col for col in ts_dispatch_lv.columns if "_ev" in col]
        hp_cols = [col for col in ts_dispatch_lv.columns if "_hp" in col]
        rh_cols = [col for col in ts_dispatch_lv.columns if "_rh" in col]
        if base_cols:
            node_dm_dict["total_base_demand_twh"] = - ts_dispatch_lv[base_cols].sum().sum() / 1e6 * int_len_h
        if ind_cols:
            node_dm_dict["total_ind_demand_twh"] = - ts_dispatch_lv[ind_cols].sum().sum() / 1e6 * int_len_h
        if solar_btm_cols:
            node_dm_dict["total_solar_btm_gen_twh"] = ts_dispatch_lv[solar_btm_cols].sum().sum() / 1e6 * int_len_h
        if ev_cols:
            node_dm_dict["total_ev_demand_twh"] = - ts_dispatch_lv[ev_cols].sum().sum() / 1e6 * int_len_h
        if hp_cols:
            node_dm_dict["total_hp_demand_twh"] = - ts_dispatch_lv[hp_cols].sum().sum() / 1e6 * int_len_h
        if rh_cols:
            node_dm_dict["total_rh_demand_twh"] = - ts_dispatch_lv[rh_cols].sum().sum() / 1e6 * int_len_h

        # Other metrics
        node_dm_dict["total_hp_capacity_gw"] = (n.links.loc[f"{node} urban central air heat pump", "p_nom_opt"] + 
                                                n.links.loc[f"{node} urban decentral air heat pump", "p_nom_opt"] +
                                                n.links.loc[f"{node} rural ground heat pump", "p_nom_opt"]) / 1e3 # Total HP capacity for the node

        # Put together in DataFrame format for later concatenation at country level
        node_km_df = pd.DataFrame(node_dm_dict, index=[0])
        country_dm_dfs.append(node_km_df)

    country_dm_df = pd.concat(country_dm_dfs, axis=0, ignore_index=True)

    return country_dm_df, n_cache, dir_cache


# Function for putting together key transport metrics for a country
def get_transport_metrics(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None):

    # Retrieve network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)

    if country not in country_dic.keys():
        raise AttributeError

    # Retrieve aggregated time series for focus buses
    ts_aggs_country, n_cache, dir_cache = ts_for_node_agg(scen_name, cy_name, ty, country, n_cache=n_cache, dir_cache=dir_cache, look_for_backup=True)

    # Set up list for country concatenation and dictionary of demand metrics
    country_tm_dfs = []

    # One set of demand metrics per node
    for node in country_dic[country]:

        node_tm_dict = {"country": country,
                "node": node,
                "total_oil_demand_twh": 0,
                "total_bev_demand_twh": 0
                }

        node_tm_dict["node"] = node
        ts_aggs_node = ts_aggs_country[node]

        # HV electricity bus
        ts_dispatch_trans = ts_aggs_node["Transport"]
        int_len_h = int((ts_dispatch_trans.index[1] - ts_dispatch_trans.index[0]).total_seconds() / 3600)
        # Load to distribution
        if "load_oil" in ts_dispatch_trans.columns:
            node_tm_dict["total_oil_demand_twh"] = - ts_dispatch_trans["load_oil"].sum() / 1e6 * int_len_h
        if "load_bev" in ts_dispatch_trans.columns:
            node_tm_dict["total_bev_demand_twh"] = - ts_dispatch_trans["load_bev"].sum() / 1e6 * int_len_h

        # Put together in DataFrame format for later concatenation at country level
        node_tm_df = pd.DataFrame(node_tm_dict, index=[0])
        country_tm_dfs.append(node_tm_df)

    country_tm_df = pd.concat(country_tm_dfs, axis=0, ignore_index=True)

    return country_tm_df, n_cache, dir_cache


# Function to retrieve electrified heat capacity with default in case of missing element
def retrieve_eheat_gen_capacity(links_df, node, eheat_techs, divider=1):

    cap = 0
    for eheat_tech in eheat_techs:
        try:
            ht_cap = links_df.loc[f"{node} {eheat_tech}", "p_nom_opt"] / divider
        except KeyError:
            ht_cap = 0
        cap += ht_cap
    
    return cap

# Function to retrieve fossil heat capacity with default in case of missing element
def retrieve_fheat_gen_capacity(links_df, node, fheat_techs, divider=1):

    cap = 0
    for fheat_tech in fheat_techs:
        try:
            ht_cap = (links_df.loc[f"{node} {fheat_tech}", "p_nom_opt"] *
                      links_df.loc[f"{node} {fheat_tech}", "efficiency"]) / divider
        except KeyError:
            ht_cap = 0
        cap += ht_cap

    return cap

# Function to retrieve CHP heat capacity with default in case of missing element
def retrieve_cheat_gen_capacity(links_df, node, fuel, divider=1):

    cap = 0
    cheat_df = links_df.loc[links_df.index.str.contains(f"{node}_{fuel}_chp"), :]
    if not cheat_df.empty:
        fuel_cap = (cheat_df["p_nom_opt"] * cheat_df["efficiency2"]).sum() / divider
        cap += fuel_cap

    return cap

# Function to retrieve electrified heat capacity with default in case of missing element
def retrieve_heat_store_capacity(stores_df, node, heat_techs, divider=1):

    cap = 0
    for heat_tech in heat_techs:
        try:
            ht_cap = stores_df.loc[f"{node} {heat_tech}", "e_nom_opt"] / divider
        except KeyError:
            ht_cap = 0
        cap += ht_cap
    
    return cap

# Function for putting together key heat metrics for a country
def get_heat_metrics(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None):

    # Retrieve network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)

    if country not in country_dic.keys():
        raise AttributeError

    # Retrieve aggregated time series for focus buses
    ts_aggs_country, n_cache, dir_cache = ts_for_node_agg(scen_name, cy_name, ty, country, n_cache=n_cache, dir_cache=dir_cache, look_for_backup=True)

    # Set up list for country concatenation and dictionary of demand metrics
    country_ht_dfs = []

    # One set of demand metrics per node
    for node in country_dic[country]:

        node_ht_dict = {"country": country,
                        "node": node,
                        "total_hp_capacity_gw": 0,
                        "total_rh_capacity_gw": 0,
                        "total_biomass_bl_capacity_gw": 0,
                        "total_gas_bl_capacity_gw": 0,
                        "total_biomass_chp_capacity_gw": 0,
                        "total_gas_chp_capacity_gw": 0,
                        "total_coal_chp_capacity_gw": 0,
                        "dsm_capacity_twh": 0,
                        "wp_capacity_twh": 0,
                        "wt_capacity_twh": 0,
                        "total_heat_demand_twh": 0,
                        "total_hp_gen_twh": 0,
                        "total_rh_gen_twh": 0,
                        "total_biomass_bl_gen_twh": 0,
                        "total_gas_bl_gen_twh": 0,
                        "total_biomass_chp_gen_twh": 0,
                        "total_gas_chp_gen_twh": 0, 
                        "total_coal_chp_gen_twh": 0,
                        }
        
        node_ht_dict["node"] = node
        ts_aggs_node = ts_aggs_country[node]

        # Heat generation capacities
        node_ht_dict["total_hp_capacity_gw"] = retrieve_eheat_gen_capacity(n.links, 
                                                                           node,
                                                                           ["urban central air heat pump", "urban decentral air heat pump", "rural air heat pump", "rural ground heat pump"], 
                                                                           divider=1e3) # Total HP capacity for the node
        node_ht_dict["total_rh_capacity_gw"] = retrieve_eheat_gen_capacity(n.links, 
                                                                           node,
                                                                           ["urban central resistive heater", "urban decentral resistive heater", "rural resistive heater"],
                                                                           divider=1e3) # Total RH capacity for the node
        node_ht_dict["total_biomass_bl_capacity_gw"] = retrieve_fheat_gen_capacity(n.links,
                                                                                   node,
                                                                                   ["urban decentral biomass boiler", "rural biomass boiler"],
                                                                                   divider=1e3) # Total biomass boiler capacity for the node
        node_ht_dict["total_gas_bl_capacity_gw"] = retrieve_fheat_gen_capacity(n.links,
                                                                               node,
                                                                               ["urban central gas boiler", "urban decentral gas boiler", "rural gas boiler"],
                                                                               divider=1e3) # Total gas boiler capacity for the node
        node_ht_dict["total_biomass_chp_capacity_gw"] = retrieve_cheat_gen_capacity(n.links, node, "biomass", divider=1e3) # Total biomass CHP capacity for the node
        node_ht_dict["total_gas_chp_capacity_gw"] = retrieve_cheat_gen_capacity(n.links, node, "gas", divider=1e3) # Total gas CHP capacity for the node
        node_ht_dict["total_coal_chp_capacity_gw"] = retrieve_cheat_gen_capacity(n.links, node, "coal", divider=1e3) # Total coal CHP capacity for the node

        # Heat storage capacities
        node_ht_dict["dsm_capacity_twh"] = retrieve_heat_store_capacity(n.stores, 
                                                                        node, 
                                                                        ["urban central heat dsm", "urban decentral heat dsm", "rural heat dsm"],
                                                                        divider=1e6) # Total demand-side management capacity for the node
        node_ht_dict["wp_capacity_twh"] = retrieve_heat_store_capacity(n.stores,
                                                                       node,
                                                                       ["urban central water pits"],
                                                                       divider=1e6) # Total water pit storage capacity for the node
        node_ht_dict["wt_capacity_twh"] = retrieve_heat_store_capacity(n.stores,
                                                                       node,
                                                                       ["urban central water tanks", "urban decentral water tanks", "rural water tanks"],
                                                                       divider=1e6) # Total water tank storage capacity for the node
        
        # Heat dispatch
        ts_dispatch_ht = ts_aggs_node["Heat"]
        int_len_h = int((ts_dispatch_ht.index[1] - ts_dispatch_ht.index[0]).total_seconds() / 3600)
        # Heat demand and supply decomposition
        load_cols = [col for col in ts_dispatch_ht.columns if "load" in col]
        hp_gen_cols = ["gen_hp"] if "gen_hp" in ts_dispatch_ht.columns else []
        rh_gen_cols = ["gen_rh"] if "gen_rh" in ts_dispatch_ht.columns else []
        biomass_bl_gen_cols = ["gen_biomass"] if "gen_biomass" in ts_dispatch_ht.columns else []
        gas_bl_gen_cols = ["gen_methane"] if "gen_methane" in ts_dispatch_ht.columns else []
        biomass_chp_gen_cols = ["gen_biomass_chp"] if "gen_biomass_chp" in ts_dispatch_ht.columns else []
        gas_chp_gen_cols = ["gen_methane_chp"] if "gen_methane_chp" in ts_dispatch_ht.columns else []
        coal_chp_gen_cols = ["gen_coal_chp"] if "gen_coal_chp" in ts_dispatch_ht.columns else []
        if load_cols:
            node_ht_dict["total_heat_demand_twh"] = - ts_dispatch_ht[load_cols].sum().sum() / 1e6 * int_len_h
        if hp_gen_cols:
            node_ht_dict["total_hp_gen_twh"] = ts_dispatch_ht[hp_gen_cols].sum().sum() / 1e6 * int_len_h
        if rh_gen_cols:
            node_ht_dict["total_rh_gen_twh"] = ts_dispatch_ht[rh_gen_cols].sum().sum() / 1e6 * int_len_h
        if biomass_bl_gen_cols:
            node_ht_dict["total_biomass_bl_gen_twh"] = ts_dispatch_ht[biomass_bl_gen_cols].sum().sum() / 1e6 * int_len_h
        if gas_bl_gen_cols:
            node_ht_dict["total_gas_bl_gen_twh"] = ts_dispatch_ht[gas_bl_gen_cols].sum().sum() / 1e6 * int_len_h
        if biomass_chp_gen_cols:
            node_ht_dict["total_biomass_chp_gen_twh"] = ts_dispatch_ht[biomass_chp_gen_cols].sum().sum() / 1e6 * int_len_h
        if gas_chp_gen_cols:
            node_ht_dict["total_gas_chp_gen_twh"] = ts_dispatch_ht[gas_chp_gen_cols].sum().sum() / 1e6 * int_len_h
        if coal_chp_gen_cols:
            node_ht_dict["total_coal_chp_gen_twh"] = ts_dispatch_ht[coal_chp_gen_cols].sum().sum() / 1e6 * int_len_h

        # Put together in DataFrame format for later concatenation at country level
        node_km_df = pd.DataFrame(node_ht_dict, index=[0])
        country_ht_dfs.append(node_km_df)

    country_ht_df = pd.concat(country_ht_dfs, axis=0, ignore_index=True)

    return country_ht_df, n_cache, dir_cache


# Function for putting together key fuel demand metrics
def get_fuel_metrics(scen_name, cy_name, ty, country, n_cache=None, dir_cache=None):

    # Retrieve network file
    n, n_cache, dir_cache = select_nf(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache)

    # Retrieve aggregated time series for focus buses
    ts_aggs_fuel, n_cache, dir_cache = ts_for_fuel_agg(scen_name, cy_name, ty, n_cache=n_cache, dir_cache=dir_cache, look_for_backup=True)
   
    # Set up list for country concatenation and dictionary of demand metrics
    fuels_dm_dfs = []

    # One demand dataframe per fuel
    for fuel, fuel_df in ts_aggs_fuel.items():
        int_len_h = int((fuel_df.index[1] - fuel_df.index[0]).total_seconds() / 3600)
        for col in fuel_df.columns:
            fuel_dm_dict = {"country": fuel, 
                            "node": country, 
                            "metric": "",
                            "value": 0}
            fuel_dm_dict["metric"] = f"{fuel} - {col} - twh"
            fuel_dm_dict["value"] = fuel_df[col].sum() / 1e6 * int_len_h
            fuels_dm_dfs.append(pd.DataFrame(fuel_dm_dict, index=[0]))

    fuels_dm_df = pd.concat(fuels_dm_dfs, axis=0, ignore_index=True)

    return fuels_dm_df, n_cache, dir_cache


# Function to compare key HV metrics across multiple scenario, climate, target year combinations, for one or more countries
def compare_key_metrics(scen_names, cy_names, tys, countries, sector_func, comparison_name="test", output_long=True):

    # Initialize cache variables
    n_cache = None
    dir_cache = None

    # Set directories and name for export
    export_dir = Path(analysis_post_dir) / "Comparison files"
    export_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    export_name = f"{timestamp}_{comparison_name} - {sector_func.__name__}"

    # Iterate over combinations (one network file at a time, to optimize computation times)
    hv_km_dfs = []
    for ty in tys:
        for cy_name in cy_names:
            for scen_name in scen_names:
                for country in countries:
                    hv_km_df, n_cache, dir_cache = sector_func(scen_name, cy_name, ty, country, n_cache=n_cache, dir_cache=dir_cache)
                    hv_km_df.insert(0, "scenario", scen_name)
                    hv_km_df.insert(0, "cy", cy_name)
                    hv_km_df.insert(0, "ty", ty)
                    hv_km_dfs.append(hv_km_df)
    hv_km_df_comp = pd.concat(hv_km_dfs, axis=0, ignore_index=True)
    if output_long:
        hv_km_df_comp = hv_km_df_comp.melt(id_vars=["ty", "cy", "scenario", "country", "node"], var_name="metric", value_name="value")
    hv_km_df_comp.to_csv(f"{export_dir}/{export_name}.csv", index=False)

    return None

In [ ]:
# Define available options for comparison
available_tys = [2030, 2040]
available_cy_names = ["central"]
available_scen_names = ["lowflex", "highflex"]
available_countries = list(country_dic.keys())

tys = [2040]
cy_names = ["central"]
scen_names = ["lowflex", "highflex"]
countries = all_countries
comparison_name = "all_bm"

# Compute key metrics for main sectors
compare_key_metrics(scen_names, cy_names, tys, ["F"], sector_func=get_fuel_metrics, comparison_name=comparison_name, output_long=False)
compare_key_metrics(scen_names, cy_names, tys, countries, sector_func=get_hv_metrics, comparison_name=comparison_name)
compare_key_metrics(scen_names, cy_names, tys, countries, sector_func=get_dm_metrics, comparison_name=comparison_name)
compare_key_metrics(scen_names, cy_names, tys, countries, sector_func=get_transport_metrics, comparison_name=comparison_name)
compare_key_metrics(scen_names, cy_names, tys, countries, sector_func=get_heat_metrics, comparison_name=comparison_name)